In [1]:
import torch

print("Torch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("Device:", "cuda" if torch.cuda.is_available() else "cpu")

Torch version: 2.10.0+cu128
CUDA available: True
Device: cuda


In [2]:
import os

print("Folders inside /kaggle/input:")
print(os.listdir("/kaggle/input"))

for folder in os.listdir("/kaggle/input"):
    folder_path = os.path.join("/kaggle/input", folder)
    print(f"\nInside: {folder_path}")
    try:
        items = os.listdir(folder_path)
        print(items[:20])
    except Exception as e:
        print("Error:", e)

Folders inside /kaggle/input:
['datasets']

Inside: /kaggle/input/datasets
['victorling', 'organizations']


In [3]:
import os
import math
import random
import torch
import torchaudio
from torch import nn
from torch.utils.data import Dataset, DataLoader, random_split
from tqdm import tqdm

SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)

SAMPLE_RATE = 16000
N_MELS = 80

BATCH_SIZE = 8
EPOCHS = 20
LR = 3e-4
BEAM_WIDTH = 10

LIBRISPEECH_PATH = "/kaggle/input/datasets/victorling/librispeech-clean/LibriSpeech/train-clean-100"

SAVE_MODEL_PATH = "/kaggle/working/best_speech_model.pth"

DATA_PERCENT = 1

Device: cuda


In [4]:
CHARS = list("abcdefghijklmnopqrstuvwxyz '")
char_to_idx = {c: i + 1 for i, c in enumerate(CHARS)}
idx_to_char = {i + 1: c for i, c in enumerate(CHARS)}

BLANK_LABEL = 0
VOCAB_SIZE = len(CHARS) + 1

def text_to_indices(text):
    text = text.lower()
    return [char_to_idx[c] for c in text if c in char_to_idx]

def indices_to_text(indices):
    return "".join(idx_to_char[i] for i in indices if i in idx_to_char)

def levenshtein(a, b):
    n = len(a)
    m = len(b)

    dp = [[0] * (m + 1) for _ in range(n + 1)]

    for i in range(n + 1):
        dp[i][0] = i
    for j in range(m + 1):
        dp[0][j] = j

    for i in range(1, n + 1):
        for j in range(1, m + 1):
            cost = 0 if a[i - 1] == b[j - 1] else 1
            dp[i][j] = min(
                dp[i - 1][j] + 1,
                dp[i][j - 1] + 1,
                dp[i - 1][j - 1] + cost
            )

    return dp[n][m]

def cer_score(reference, prediction):
    ref = list(reference)
    pred = list(prediction)

    if len(ref) == 0:
        return 0.0 if len(pred) == 0 else 1.0

    return levenshtein(ref, pred) / len(ref)

def wer_score(reference, prediction):
    ref_words = reference.split()
    pred_words = prediction.split()

    if len(ref_words) == 0:
        return 0.0 if len(pred_words) == 0 else 1.0

    return levenshtein(ref_words, pred_words) / len(ref_words)

In [5]:
class MyLibriSpeechDataset(Dataset):
    def __init__(self, root_folder):
        self.samples = []

        for speaker_id in os.listdir(root_folder):
            speaker_path = os.path.join(root_folder, speaker_id)
            if not os.path.isdir(speaker_path):
                continue

            for chapter_id in os.listdir(speaker_path):
                chapter_path = os.path.join(speaker_path, chapter_id)
                if not os.path.isdir(chapter_path):
                    continue

                transcript_file = os.path.join(chapter_path, f"{speaker_id}-{chapter_id}.trans.txt")
                if not os.path.exists(transcript_file):
                    continue

                transcript_dict = {}
                with open(transcript_file, "r", encoding="utf-8") as f:
                    for line in f:
                        parts = line.strip().split(" ", 1)
                        if len(parts) == 2:
                            file_id, text = parts
                            transcript_dict[file_id] = text.lower()

                for file_name in os.listdir(chapter_path):
                    if file_name.endswith(".flac"):
                        file_id = file_name.replace(".flac", "")
                        file_path = os.path.join(chapter_path, file_name)

                        if file_id in transcript_dict:
                            self.samples.append((file_path, transcript_dict[file_id]))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        file_path, transcript = self.samples[idx]

        waveform, sr = torchaudio.load(file_path)

        if sr != SAMPLE_RATE:
            waveform = torchaudio.functional.resample(waveform, sr, SAMPLE_RATE)

        label = torch.tensor(text_to_indices(transcript), dtype=torch.long)

        return waveform, transcript, label

In [6]:
mel_transform = torchaudio.transforms.MelSpectrogram(
    sample_rate=SAMPLE_RATE,
    n_mels=N_MELS,
    n_fft=400,
    win_length=400,
    hop_length=160
)

db_transform = torchaudio.transforms.AmplitudeToDB()

time_mask = torchaudio.transforms.TimeMasking(time_mask_param=30)
freq_mask = torchaudio.transforms.FrequencyMasking(freq_mask_param=10)

def normalize_spec(spec):
    mean = spec.mean()
    std = spec.std()
    return (spec - mean) / (std + 1e-5)

def augment_waveform(waveform):
    if random.random() < 0.3:
        noise = torch.randn_like(waveform) * 0.003
        waveform = waveform + noise

    if random.random() < 0.3:
        gain = random.uniform(0.9, 1.1)
        waveform = waveform * gain

    return waveform

def collate_fn(batch, apply_specaug=False):
    specs = []
    input_lengths = []
    labels = []
    label_lengths = []
    transcripts = []

    for waveform, transcript, label in batch:
        waveform = waveform.squeeze(0)

        if apply_specaug:
            waveform = augment_waveform(waveform)

        spec = mel_transform(waveform)
        spec = db_transform(spec)
        spec = normalize_spec(spec)

        time_steps = spec.shape[1]
        input_lengths.append(time_steps)

        if apply_specaug:
            spec = spec.unsqueeze(0)
            spec = time_mask(spec)
            spec = freq_mask(spec)
            spec = spec.squeeze(0)

        specs.append(spec)
        labels.append(label)
        label_lengths.append(len(label))
        transcripts.append(transcript)

    max_time = max(input_lengths)

    padded_specs = []
    for spec in specs:
        pad_amount = max_time - spec.shape[1]
        if pad_amount > 0:
            spec = nn.functional.pad(spec, (0, pad_amount))
        padded_specs.append(spec)

    padded_specs = torch.stack(padded_specs)
    labels = torch.cat(labels)

    input_lengths = torch.tensor(input_lengths, dtype=torch.long)
    label_lengths = torch.tensor(label_lengths, dtype=torch.long)

    return padded_specs, labels, input_lengths, label_lengths, transcripts

In [7]:
class SpeechModel(nn.Module):
    def __init__(self):
        super().__init__()

        self.cnn = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.MaxPool2d((2, 1)),

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.MaxPool2d((2, 1)),

            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.MaxPool2d((2, 1))
        )

        self.lstm_input_size = 128 * 10

        self.bilstm = nn.LSTM(
            input_size=self.lstm_input_size,
            hidden_size=256,
            num_layers=3,
            dropout=0.3,
            bidirectional=True,
            batch_first=True
        )

        self.dropout = nn.Dropout(0.3)
        self.fc = nn.Linear(512, VOCAB_SIZE)

    def forward(self, x):
        x = x.unsqueeze(1)
        x = self.cnn(x)

        b, c, f, t = x.size()

        x = x.permute(0, 3, 1, 2)
        x = x.reshape(b, t, c * f)

        x, _ = self.bilstm(x)
        x = self.dropout(x)
        x = self.fc(x)
        x = x.log_softmax(dim=2)

        return x

In [8]:
def greedy_decode(log_probs, blank=0):
    pred_ids = torch.argmax(log_probs, dim=2)
    results = []

    for seq in pred_ids:
        decoded = []
        prev = blank

        for idx in seq.tolist():
            if idx != blank and idx != prev:
                decoded.append(idx)
            prev = idx

        results.append(indices_to_text(decoded))

    return results

def log_sum_exp(a, b):
    if a == -float("inf"):
        return b
    if b == -float("inf"):
        return a
    if a > b:
        return a + math.log1p(math.exp(b - a))
    else:
        return b + math.log1p(math.exp(a - b))

def ctc_beam_search_single(log_probs, beam_width=10, blank=0):
    beams = {("", blank): 0.0}

    for t in range(log_probs.size(0)):
        new_beams = {}

        for (prefix, last_char), score in beams.items():
            for c in range(log_probs.size(1)):
                new_score = score + log_probs[t, c].item()

                if c == blank:
                    key = (prefix, blank)
                    if key not in new_beams:
                        new_beams[key] = new_score
                    else:
                        new_beams[key] = log_sum_exp(new_beams[key], new_score)
                else:
                    char = idx_to_char.get(c, "")
                    if c == last_char:
                        new_prefix = prefix
                    else:
                        new_prefix = prefix + char

                    key = (new_prefix, c)
                    if key not in new_beams:
                        new_beams[key] = new_score
                    else:
                        new_beams[key] = log_sum_exp(new_beams[key], new_score)

        sorted_beams = sorted(new_beams.items(), key=lambda x: x[1], reverse=True)
        beams = dict(sorted_beams[:beam_width])

    best_prefix = max(beams.items(), key=lambda x: x[1])[0][0]
    return best_prefix

def beam_decode(log_probs, beam_width=10, blank=0):
    results = []
    for i in range(log_probs.size(0)):
        text = ctc_beam_search_single(log_probs[i], beam_width=beam_width, blank=blank)
        results.append(text)
    return results

In [9]:
def evaluate(model, loader, criterion, use_beam=False, max_batches=None):
    model.eval()

    total_loss = 0.0
    total_cer = 0.0
    total_wer = 0.0
    total_count = 0

    example_refs = []
    example_preds = []

    with torch.no_grad():
        for batch_idx, (specs, labels, input_lengths, label_lengths, transcripts) in enumerate(loader):
            specs = specs.to(DEVICE)
            labels = labels.to(DEVICE)

            outputs = model(specs)
            output_lengths = input_lengths.to(DEVICE)

            loss = criterion(
                outputs.permute(1, 0, 2),
                labels,
                output_lengths,
                label_lengths.to(DEVICE)
            )

            total_loss += loss.item()

            if use_beam:
                preds = beam_decode(outputs.cpu(), beam_width=BEAM_WIDTH, blank=BLANK_LABEL)
            else:
                preds = greedy_decode(outputs.cpu(), blank=BLANK_LABEL)

            for ref, pred in zip(transcripts, preds):
                total_cer += cer_score(ref, pred)
                total_wer += wer_score(ref, pred)
                total_count += 1

            if len(example_refs) < 3:
                for ref, pred in zip(transcripts, preds):
                    if len(example_refs) < 3:
                        example_refs.append(ref)
                        example_preds.append(pred)

            if max_batches is not None and (batch_idx + 1) >= max_batches:
                break

    avg_loss = total_loss / (batch_idx + 1)
    avg_cer = total_cer / total_count
    avg_wer = total_wer / total_count

    return avg_loss, avg_cer, avg_wer, example_refs, example_preds

In [10]:
full_dataset = MyLibriSpeechDataset(LIBRISPEECH_PATH)
print("Total samples:", len(full_dataset))

if DATA_PERCENT < 1.0:
    small_size = int(len(full_dataset) * DATA_PERCENT)
    full_dataset, _ = random_split(
        full_dataset,
        [small_size, len(full_dataset) - small_size],
        generator=torch.Generator().manual_seed(SEED)
    )

train_size = int(0.9 * len(full_dataset))
val_size = len(full_dataset) - train_size

train_dataset, val_dataset = random_split(
    full_dataset,
    [train_size, val_size],
    generator=torch.Generator().manual_seed(SEED)
)

print("Train samples:", len(train_dataset))
print("Val samples:", len(val_dataset))

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=lambda batch: collate_fn(batch, apply_specaug=True),
    num_workers=2,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=lambda batch: collate_fn(batch, apply_specaug=False),
    num_workers=2,
    pin_memory=True
)

Total samples: 28539
Train samples: 25685
Val samples: 2854


In [11]:
model = SpeechModel().to(DEVICE)

criterion = nn.CTCLoss(blank=BLANK_LABEL, zero_infinity=True)

optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="min",
    factor=0.5,
    patience=2
)

print(model)

SpeechModel(
  (cnn): Sequential(
    (0): Conv2d(1, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
    (3): Dropout(p=0.1, inplace=False)
    (4): MaxPool2d(kernel_size=(2, 1), stride=(2, 1), padding=0, dilation=1, ceil_mode=False)
    (5): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (6): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (7): ReLU()
    (8): Dropout(p=0.1, inplace=False)
    (9): MaxPool2d(kernel_size=(2, 1), stride=(2, 1), padding=0, dilation=1, ceil_mode=False)
    (10): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (11): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (12): ReLU()
    (13): Dropout(p=0.1, inplace=False)
    (14): MaxPool2d(kernel_size=(2, 1), stride=(2, 1), padding=0, dilation=1, ceil_mode=False)
  )
  (bilstm): LSTM(

In [ ]:
best_val_cer = float("inf")

for epoch in range(EPOCHS):
    model.train()
    total_train_loss = 0.0

    progress_bar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS}")

    for specs, labels, input_lengths, label_lengths, transcripts in progress_bar:
        specs = specs.to(DEVICE)
        labels = labels.to(DEVICE)

        optimizer.zero_grad()

        outputs = model(specs)
        output_lengths = input_lengths.to(DEVICE)

        loss = criterion(
            outputs.permute(1, 0, 2),
            labels,
            output_lengths,
            label_lengths.to(DEVICE)
        )

        if torch.isnan(loss):
            print("NaN loss found, skipping batch")
            continue

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
        optimizer.step()

        total_train_loss += loss.item()
        progress_bar.set_postfix(loss=loss.item())

    avg_train_loss = total_train_loss / len(train_loader)

    val_loss, val_cer, val_wer, refs, preds = evaluate(
        model,
        val_loader,
        criterion,
        use_beam=False,
        max_batches=50
    )

    scheduler.step(val_loss)

    print(f"\nEpoch {epoch+1}/{EPOCHS}")
    print(f"Train Loss: {avg_train_loss:.4f}")
    print(f"Val Loss:   {val_loss:.4f}")
    print(f"Val CER:    {val_cer:.4f}")
    print(f"Val WER:    {val_wer:.4f}")

    print("\nSample predictions:")
    for i in range(len(refs)):
        print("GT  :", refs[i])
        print("PRED:", preds[i])
        print("-" * 50)

    if val_cer < best_val_cer:
        best_val_cer = val_cer
        torch.save(model.state_dict(), SAVE_MODEL_PATH)
        print("Best model saved to:", SAVE_MODEL_PATH)

Epoch 1/20: 100%|██████████| 3211/3211 [24:08<00:00,  2.22it/s, loss=1.12] 



Epoch 1/20
Train Loss: 1.9269
Val Loss:   0.9591
Val CER:    0.3071
Val WER:    0.7561

Sample predictions:
GT  : now as there had none died in the city for all this time my lord mayor gave certificates of health without any difficulty to all those who lived in the ninety seven parishes and to those within the liberties too
PRED: no ist  ther had non didin iity for althistme my lord mar gav so to icas of het with ot anin tof atote to al dhos wolived in the nine seven pearthios and di tose with ed theliverdes to
--------------------------------------------------
GT  : i ambled up put a foot on the hub of a wheel and said i simply want to say it's a cold day you as soon as he had finished i said by way of civil explanation
PRED: ha imblo ho pout af fo on the hav ove we o an sad  shimly wo to sa is the co va o man sen is thead finchd  shad bywa o a sive i peation
--------------------------------------------------
GT  : frightened let me pass prisoner the whole deck burst into a great lau

Epoch 2/20: 100%|██████████| 3211/3211 [24:03<00:00,  2.22it/s, loss=0.809]



Epoch 2/20
Train Loss: 0.9239
Val Loss:   0.6771
Val CER:    0.2166
Val WER:    0.6080

Sample predictions:
GT  : now as there had none died in the city for all this time my lord mayor gave certificates of health without any difficulty to all those who lived in the ninety seven parishes and to those within the liberties too
PRED: not as ther had non died in isity for al thistme my lard mare gave so tif icans of helt without ini divaculty to al those wo lived in the nindy seven parishes in de tose with an the liberdis to
--------------------------------------------------
GT  : i ambled up put a foot on the hub of a wheel and said i simply want to say it's a cold day you as soon as he had finished i said by way of civil explanation
PRED: hambedot put af fot on the have of a weal an sad  shimly wat to sa as acol va yo ess inis he had finish i sad by ay of a cive espiation
--------------------------------------------------
GT  : frightened let me pass prisoner the whole deck burst into a 

Epoch 3/20: 100%|██████████| 3211/3211 [24:09<00:00,  2.22it/s, loss=0.852]



Epoch 3/20
Train Loss: 0.7358
Val Loss:   0.5617
Val CER:    0.1796
Val WER:    0.5196

Sample predictions:
GT  : now as there had none died in the city for all this time my lord mayor gave certificates of health without any difficulty to all those who lived in the ninety seven parishes and to those within the liberties too
PRED: no as ther had non did in i city for al this time my lord mar gave so tificats of helt without anydificulty to al those wo lived in the niny seven parishes ind to those with ad the libertis to
--------------------------------------------------
GT  : i ambled up put a foot on the hub of a wheel and said i simply want to say it's a cold day you as soon as he had finished i said by way of civil explanation
PRED: haimble a putta foot on the havbiva weal and sad i simly wat to say its hecol day you issinis he ad finish i said byway f a civil exliation
--------------------------------------------------
GT  : frightened let me pass prisoner the whole deck burst into

Epoch 4/20:  34%|███▍      | 1089/3211 [08:14<16:07,  2.19it/s, loss=0.551]

In [ ]:
print("Loading best model...")
model.load_state_dict(torch.load(SAVE_MODEL_PATH, map_location=DEVICE))

final_loss, final_cer, final_wer, refs, preds = evaluate(
    model,
    val_loader,
    criterion,
    use_beam=True,
    max_batches=50
)

print("\nFinal Validation Results (Beam Search)")
print(f"Loss: {final_loss:.4f}")
print(f"CER : {final_cer:.4f}")
print(f"WER : {final_wer:.4f}")

print("\nFinal sample predictions:")
for i in range(len(refs)):
    print("GT  :", refs[i])
    print("PRED:", preds[i])
    print("-" * 50)

In [ ]:
model.eval()

sample_waveform, sample_text, _ = full_dataset[0]

with torch.no_grad():
    spec = mel_transform(sample_waveform.squeeze(0))
    spec = db_transform(spec)
    spec = normalize_spec(spec)
    spec = spec.unsqueeze(0).to(DEVICE)

    output = model(spec)

greedy_pred = greedy_decode(output.cpu(), blank=BLANK_LABEL)[0]
beam_pred = beam_decode(output.cpu(), beam_width=BEAM_WIDTH, blank=BLANK_LABEL)[0]

print("Ground Truth :", sample_text)
print("Greedy Pred  :", greedy_pred)
print("Beam Pred    :", beam_pred)

In [ ]:
# =========================================================
# Test the trained model on Mozilla Common Voice
# =========================================================

import os
import csv
import torchaudio
from torch.utils.data import Dataset, DataLoader

# CHANGE THIS PATH to your actual Common Voice folder
COMMONVOICE_PATH = "/kaggle/input/datasets/organizations/mozillaorg/common-voice"

# choose which split to test on
TSV_FILE = os.path.join(COMMONVOICE_PATH, "test.tsv")
if not os.path.exists(TSV_FILE):
    TSV_FILE = os.path.join(COMMONVOICE_PATH, "validated.tsv")

CLIPS_PATH = os.path.join(COMMONVOICE_PATH, "clips")

print("Using TSV file:", TSV_FILE)
print("Using clips path:", CLIPS_PATH)


class CommonVoiceDataset(Dataset):
    def __init__(self, tsv_file, clips_path, max_samples=None):
        self.samples = []
        self.clips_path = clips_path

        with open(tsv_file, "r", encoding="utf-8") as f:
            reader = csv.DictReader(f, delimiter="\t")

            for row in reader:
                if "path" not in row or "sentence" not in row:
                    continue

                rel_path = row["path"].strip()
                sentence = row["sentence"].strip().lower()

                if len(sentence) == 0:
                    continue

                # keep only characters supported by your model
                filtered_sentence = "".join([c for c in sentence if c in char_to_idx])

                if len(filtered_sentence) == 0:
                    continue

                audio_path = os.path.join(clips_path, rel_path)

                if os.path.exists(audio_path):
                    self.samples.append((audio_path, filtered_sentence))

                if max_samples is not None and len(self.samples) >= max_samples:
                    break

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        file_path, transcript = self.samples[idx]

        waveform, sr = torchaudio.load(file_path)

        # convert to mono if needed
        if waveform.shape[0] > 1:
            waveform = waveform.mean(dim=0, keepdim=True)

        # resample to match training
        if sr != SAMPLE_RATE:
            waveform = torchaudio.functional.resample(waveform, sr, SAMPLE_RATE)

        label = torch.tensor(text_to_indices(transcript), dtype=torch.long)

        return waveform, transcript, label


# create test dataset
cv_test_dataset = CommonVoiceDataset(
    tsv_file=TSV_FILE,
    clips_path=CLIPS_PATH,
    max_samples=500   # you can increase this or set to None
)

print("Common Voice test samples:", len(cv_test_dataset))

cv_test_loader = DataLoader(
    cv_test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=lambda batch: collate_fn(batch, apply_specaug=False),
    num_workers=2,
    pin_memory=True
)

# evaluate
cv_loss, cv_cer, cv_wer, refs, preds = evaluate(
    model,
    cv_test_loader,
    criterion,
    use_beam=True,
    max_batches=None
)

print("\nCommon Voice Test Results")
print(f"Loss: {cv_loss:.4f}")
print(f"CER : {cv_cer:.4f}")
print(f"WER : {cv_wer:.4f}")

print("\nSample predictions:")
for i in range(min(5, len(refs))):
    print("GT  :", refs[i])
    print("PRED:", preds[i])
    print("-" * 50)